# PoliMillionaire Python-assisted math solver

This notebook answers multiple-choice math/statistics questions with a compact pipeline:

1. safe model-generated Python for computation-heavy questions
2. reasoning solver for conceptual questions
3. structured model answer fallback
4. strict letter-only fallback when needed


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import json
import math
import textwrap
from getpass import getpass

import torch
import sympy as sp

In [ ]:
BASE_DIR_CANDIDATES = [
    '/content/gdrive/MyDrive/NLP_assignment1',
    '/content/gdrive/MyDrive/NLP_assignment',
    '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment',
]
BASE_DIR = next((candidate for candidate in BASE_DIR_CANDIDATES if os.path.exists(candidate)), BASE_DIR_CANDIDATES[0])
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR:', BASE_DIR)
print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('Could not find the NLP assignment folder in Google Drive. Update BASE_DIR_CANDIDATES.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

print('Path added successfully.')

In [ ]:
%pip install -q --upgrade "transformers==4.51.3" "accelerate==1.3.0" "huggingface_hub>=0.30.0,<1.0" sentencepiece protobuf sympy "bitsandbytes>=0.46.1,<0.49"

import sys
import importlib.metadata as metadata

print('transformers:', metadata.version('transformers'))
print('accelerate:', metadata.version('accelerate'))
print('huggingface_hub:', metadata.version('huggingface_hub'))

if 'transformers' in sys.modules:
    print('Note: transformers was already imported. If model loading fails, restart the kernel manually and run from the top.')

In [ ]:
import importlib.metadata as metadata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

print('Using transformers:', metadata.version('transformers'))

In [ ]:
API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())
USERNAME = (__import__("os").getenv("POLI_MILLIONAIRE_USERNAME") or input("PoliMillionaire username: ").strip())
PASSWORD = (__import__("os").getenv("POLI_MILLIONAIRE_PASSWORD") or __import__("getpass").getpass("PoliMillionaire password: ").strip())

if PASSWORD is None:
    PASSWORD = getpass('Millionaire password: ')

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

In [ ]:
competitions = client.competitions.list_all()
for competition in competitions:
    print(competition.id, competition.name, competition.max_levels)

COMPETITION_ID = 3
print('Selected competition:', COMPETITION_ID)

In [ ]:
import os
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN = ''
if HF_TOKEN == 'PASTE_YOUR_HF_TOKEN_HERE' or not HF_TOKEN.strip():
    raise ValueError('Paste your Hugging Face token into HF_TOKEN before loading Llama.')

login(token=HF_TOKEN, add_to_git_credential=False)

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
MAX_PROMPT_TOKENS = int(os.getenv('MILLIONAIRE_MAX_PROMPT_TOKENS', '1024'))
DEFAULT_MAX_NEW_TOKENS = int(os.getenv('MILLIONAIRE_MAX_NEW_TOKENS', '96'))
OFFLOAD_DIR = os.getenv('MILLIONAIRE_OFFLOAD_DIR', '/content/llama_offload')
os.makedirs(OFFLOAD_DIR, exist_ok=True)
DEFAULT_TEMPERATURE = float(os.getenv('MILLIONAIRE_TEMPERATURE', '0.2'))
DEFAULT_TOP_P = float(os.getenv('MILLIONAIRE_TOP_P', '0.9'))

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    print('CUDA device:', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA GPU not detected. Llama 8B 4-bit is intended for a GPU runtime such as Colab T4.')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading model:', model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_memory = {0: os.getenv('MILLIONAIRE_GPU_MAX_MEMORY', '13GiB'), 'cpu': os.getenv('MILLIONAIRE_CPU_MAX_MEMORY', '24GiB')}

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map='auto',
    max_memory=max_memory,
    offload_folder=OFFLOAD_DIR,
    offload_state_dict=True,
    offload_buffers=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)
model.eval()
if hasattr(model.config, 'use_cache'):
    model.config.use_cache = True


def make_prompt(user_message):
    messages = [{'role': 'user', 'content': user_message}]
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return user_message


def generate_text(user_message, max_new_tokens=None, temperature=None):
    max_new_tokens = DEFAULT_MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens
    temperature = DEFAULT_TEMPERATURE if temperature is None else temperature
    prompt = make_prompt(user_message)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    ).to(model.device)
    generation_kwargs = {
        'max_new_tokens': max_new_tokens,
        'pad_token_id': tokenizer.eos_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'do_sample': temperature > 0,
        'use_cache': True,
    }
    if temperature > 0:
        generation_kwargs['temperature'] = temperature
        generation_kwargs['top_p'] = DEFAULT_TOP_P

    with torch.inference_mode():
        outputs = model.generate(**inputs, **generation_kwargs)

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [ ]:
def extract_json_objects(text):
    text = str(text or '').strip()
    fence = chr(96) * 3
    if text.startswith(fence):
        text = text[len(fence):].strip()
        if text.lower().startswith('json'):
            text = text[4:].strip()
    if text.endswith(fence):
        text = text[:-len(fence)].strip()

    decoder = json.JSONDecoder()
    objects = []
    for index, char in enumerate(text):
        if char != '{':
            continue
        try:
            value, _ = decoder.raw_decode(text[index:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            objects.append(value)
    return objects


def extract_last_json_object(text):
    objects = extract_json_objects(text)
    for value in reversed(objects):
        if isinstance(value, dict) and 'answer' in value:
            return value
    return objects[-1] if objects else None


def extract_letter(text):
    text = str(text or '').strip().upper()
    match = re.search(r'\b([ABCD])\b', text)
    if match:
        return match.group(1)
    return text[0] if text and text[0] in 'ABCD' else None


def question_to_text(question):
    lines = [str(question.text).strip()]
    for index, option in enumerate(question.options[:4]):
        lines.append(f'{chr(65 + index)}) {option.text}')
    return '\n'.join(lines)


def option_id_for_letter(question, letter):
    index = ['A', 'B', 'C', 'D'].index(letter)
    return question.options[index].id


In [ ]:
import ast as py_ast
import contextlib
import io
import html
import itertools
import statistics
import urllib.parse
import urllib.request
from fractions import Fraction

MODEL_TEMPERATURE = float(os.getenv('MILLIONAIRE_TEMPERATURE', '0.2'))
STRUCTURED_TOKENS = int(os.getenv('MILLIONAIRE_STRUCTURED_TOKENS', '192'))
REASONING_TOKENS = int(os.getenv('MILLIONAIRE_REASONING_TOKENS', '320'))
PYTHON_SOLVER_TOKENS = int(os.getenv('MILLIONAIRE_PYTHON_SOLVER_TOKENS', '384'))
LETTER_TOKENS = int(os.getenv('MILLIONAIRE_LETTER_TOKENS', '8'))
USE_PYTHON_SOLVER = os.getenv('MILLIONAIRE_USE_PYTHON_SOLVER', '1') != '0'
USE_WIKIPEDIA_TOOL = os.getenv('MILLIONAIRE_USE_WIKIPEDIA_TOOL', '0') == '1'
USE_REASONING_SOLVER = os.getenv('MILLIONAIRE_USE_REASONING_SOLVER', '1') != '0'
USE_STRUCTURED_LLM = os.getenv('MILLIONAIRE_USE_STRUCTURED_LLM', '1') != '0'
WIKIPEDIA_QUERY_TOKENS = int(os.getenv('MILLIONAIRE_WIKIPEDIA_QUERY_TOKENS', '64'))
WIKIPEDIA_TIMEOUT_SECONDS = float(os.getenv('MILLIONAIRE_WIKIPEDIA_TIMEOUT_SECONDS', '2.5'))
WIKIPEDIA_MAX_SNIPPETS = int(os.getenv('MILLIONAIRE_WIKIPEDIA_MAX_SNIPPETS', '2'))
WIKIPEDIA_MAX_CONTEXT_CHARS = int(os.getenv('MILLIONAIRE_WIKIPEDIA_MAX_CONTEXT_CHARS', '1200'))

LETTERS = ['A', 'B', 'C', 'D']


def normalize_text(value):
    text = str(value).lower()
    text = text.replace(chr(8722), '-')
    text = text.replace(chr(8211), '-')
    text = text.replace(chr(8212), '-')
    text = text.replace('?', '<=')
    text = text.replace('?', '>=')
    text = text.replace('$', '')
    return re.sub(r'\s+', ' ', text).strip()






def option_texts(question):
    return {LETTERS[i]: str(option.text) for i, option in enumerate(question.options[:4])}










def option_id_for_letter_safe(question, letter):
    return option_id_for_letter(question, letter if letter in LETTERS else 'A')


def numeric_option_values(question):
    values = {}
    for letter, text in option_texts(question).items():
        match = re.search(r'[-+]?\d+(?:\.\d+)?', text.replace(',', ''))
        if match:
            values[letter] = float(match.group(0))
    return values


def parse_option_math_value(text):
    expr = normalize_text(text)
    expr = expr.replace('?', 'pi')
    expr = expr.replace('^', '**')
    expr = re.sub(r'(\d)\s*sqrt', r'\1*sqrt', expr)
    expr = re.sub(r'(\d)\s*pi', r'\1*pi', expr)
    expr = expr.replace(' ', '')
    if not re.fullmatch(r'[0-9.+\-*/()sqrtpi]+', expr):
        return None
    try:
        value = sp.N(sp.sympify(expr, locals={'sqrt': sp.sqrt, 'pi': sp.pi}), 12)
    except Exception:
        return None
    if getattr(value, 'free_symbols', set()):
        return None
    return float(value)


def math_option_values(question):
    values = {}
    for letter, text in option_texts(question).items():
        value = parse_option_math_value(text)
        if value is not None:
            values[letter] = value
    return values


def parse_complex_like(value):
    expr = str(value).strip().lower().replace(' ', '').replace('i', 'j')
    expr = expr.replace('−', '-').replace('–', '-').replace('—', '-')
    expr = re.sub(r'(^|[+\-])j', r'\g<1>1j', expr)
    if not re.fullmatch(r'[0-9.+\-()j]+', expr) or 'j' not in expr:
        return None
    try:
        return complex(expr)
    except Exception:
        return None


def complex_option_values(question):
    values = {}
    for letter, text in option_texts(question).items():
        value = parse_complex_like(text)
        if value is not None:
            values[letter] = value
    return values


def closest_option_by_complex(question, target):
    values = complex_option_values(question)
    if not values:
        return None
    target = complex(target)
    return min(values, key=lambda letter: abs(values[letter] - target))


def closest_option_by_value(question, target):
    values = math_option_values(question) or numeric_option_values(question)
    if not values:
        return None
    return min(values, key=lambda letter: abs(values[letter] - float(target)))






























def build_python_solver_prompt(question):
    return f'''You may solve this multiple-choice math/statistics question by writing short Python code.
Return only raw Python code, or return exactly NO_PYTHON if code would not help. Do not use Markdown. Do not explain.

Code rules:
- Assign the final computed value to a variable named answer.
- Do not import anything. Available names include math, itertools, statistics, Fraction, sp, comb, sqrt, range, sum, min, max.
- Do not use files, network, input, eval, exec, open, or private names.
- Keep code under 12 lines.
- Compute the mathematical result, not just the option letter, unless the result is inherently categorical.

Question:
{question_to_text(question)}

Python code:'''


def extract_python_code(raw_output):
    text = str(raw_output or '').strip()
    request = extract_last_json_object(text)
    if isinstance(request, dict) and request.get('code'):
        return str(request.get('code')).strip()

    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE).strip()
    if re.fullmatch(r'NO_PYTHON', text, flags=re.IGNORECASE):
        return None

    fence = re.search(r'```(?:python)?\s*(.*?)```', text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        text = fence.group(1).strip()
    text = re.sub(r'^```(?:python)?\s*', '', text, flags=re.IGNORECASE).strip()
    text = re.sub(r'\s*```\s*$', '', text).strip()
    if 'NO_PYTHON' in text.upper() and 'answer' not in text:
        return None

    candidates = [text]
    lines = [line.rstrip() for line in text.splitlines()]
    for start in range(len(lines)):
        candidate = '\n'.join(lines[start:]).strip()
        if 'answer' in candidate:
            candidates.append(candidate)
    for end in range(len(lines), 0, -1):
        candidate = '\n'.join(lines[:end]).strip()
        if 'answer' in candidate:
            candidates.append(candidate)

    seen = set()
    for candidate in candidates:
        candidate = candidate.strip()
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        try:
            validate_generated_python(candidate)
            return candidate
        except Exception:
            continue
    return text or None


def validate_generated_python(code):
    code = str(code).strip()
    if not code or len(code) > 1600:
        raise ValueError('Generated code is empty or too long.')
    banned = [
        '__', 'import ', 'from ', 'open(', 'exec(', 'eval(', 'compile(', 'input(',
        'globals(', 'locals(', 'vars(', 'dir(', 'getattr(', 'setattr(', 'delattr(',
        'os.', 'sys.', 'subprocess', 'socket', 'requests', 'urllib', 'pathlib',
        'shutil', 'pickle', 'marshal', 'ctypes', 'multiprocessing', 'threading',
    ]
    lowered = code.lower()
    for marker in banned:
        if marker in lowered:
            raise ValueError(f'Generated code contains banned pattern: {marker}')

    tree = py_ast.parse(code, mode='exec')
    banned_nodes = (
        py_ast.Import, py_ast.ImportFrom, py_ast.With, py_ast.AsyncWith,
        py_ast.FunctionDef, py_ast.AsyncFunctionDef, py_ast.ClassDef,
        py_ast.Global, py_ast.Nonlocal, py_ast.Delete, py_ast.Try,
        py_ast.Raise, py_ast.While,
    )
    for node in py_ast.walk(tree):
        if isinstance(node, banned_nodes):
            raise ValueError(f'Generated code uses disallowed syntax: {type(node).__name__}')
        if isinstance(node, py_ast.Attribute) and str(node.attr).startswith('_'):
            raise ValueError('Generated code uses private attributes.')
        if isinstance(node, py_ast.Name) and str(node.id).startswith('_'):
            raise ValueError('Generated code uses private names.')
    return code


def execute_generated_python(code, result_variable='answer'):
    code = validate_generated_python(code)
    safe_builtins = {
        'abs': abs, 'all': all, 'any': any, 'bool': bool, 'dict': dict,
        'enumerate': enumerate, 'float': float, 'int': int, 'len': len,
        'list': list, 'max': max, 'min': min, 'pow': pow, 'range': range,
        'round': round, 'set': set, 'sorted': sorted, 'str': str,
        'sum': sum, 'tuple': tuple, 'zip': zip, 'print': print,
    }
    safe_globals = {
        '__builtins__': safe_builtins,
        'math': math,
        'itertools': itertools,
        'statistics': statistics,
        'Fraction': Fraction,
        'sp': sp,
        'comb': math.comb,
        'factorial': math.factorial,
        'sqrt': math.sqrt,
        'floor': math.floor,
        'ceil': math.ceil,
    }
    exec_env = dict(safe_globals)
    stdout_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer):
        exec(compile(code, '<generated_solver>', 'exec'), exec_env, exec_env)
    stdout = stdout_buffer.getvalue().strip()
    result_name = str(result_variable or 'answer').strip()
    result = exec_env.get(result_name, None)
    if result is None and 'answer' in exec_env:
        result = exec_env['answer']
    if result is None and stdout:
        result = stdout.splitlines()[-1].strip()
    if result is None:
        raise ValueError('Generated code did not set an answer variable or print a result.')
    user_locals = {k: str(v) for k, v in exec_env.items() if k not in safe_globals and not k.startswith('_')}
    return {'code': code, 'stdout': stdout, 'result': result, 'locals': user_locals}


def letter_from_python_result(question, result):
    if isinstance(result, dict):
        if 'answer' in result:
            nested = letter_from_python_result(question, result['answer'])
            if nested:
                return nested
        if 'value' in result:
            nested = letter_from_python_result(question, result['value'])
            if nested:
                return nested
    if isinstance(result, str):
        strict = extract_strict_letter_output(result)
        if strict in LETTERS:
            return strict
        complex_value = parse_complex_like(result)
        if complex_value is not None:
            complex_letter = closest_option_by_complex(question, complex_value)
            if complex_letter:
                return complex_letter
        try:
            numeric = float(result)
        except ValueError:
            numeric = None
        if numeric is not None:
            return closest_option_by_value(question, numeric)
        normalized_result = normalize_text(result)
        for letter, text in option_texts(question).items():
            if normalize_text(text) == normalized_result:
                return letter
        return None
    if isinstance(result, (int, float)):
        return closest_option_by_value(question, float(result))
    if isinstance(result, complex):
        return closest_option_by_complex(question, result)
    if hasattr(result, 'evalf'):
        try:
            return closest_option_by_value(question, float(sp.N(result, 12)))
        except Exception:
            return None
    return None


def run_python_solver(question):
    raw_output = generate_text(build_python_solver_prompt(question), max_new_tokens=PYTHON_SOLVER_TOKENS, temperature=MODEL_TEMPERATURE)
    code = extract_python_code(raw_output)
    if not code:
        return {'ok': False, 'answer': None, 'reason': 'Python solver skipped or produced no usable code.', 'raw_output': raw_output}
    request = {'needs_python': True, 'code': code, 'result_variable': 'answer'}
    result_variable = 'answer'
    try:
        execution = execute_generated_python(code, result_variable)
        letter = letter_from_python_result(question, execution['result'])
    except Exception as exc:
        return {'ok': False, 'answer': None, 'reason': f'Python execution failed: {type(exc).__name__}: {exc}', 'raw_output': raw_output, 'request': request}
    if letter not in LETTERS:
        return {'ok': False, 'answer': None, 'reason': 'Python result did not match any option.', 'raw_output': raw_output, 'request': request, 'execution': execution}
    return {
        'ok': True,
        'answer': letter,
        'confidence': 'high',
        'reason': 'Computed with generated Python and matched to the closest answer choice.',
        'source': 'llm:python_solver',
        'raw_output': raw_output,
        'request': request,
        'execution': execution,
    }


def strip_private_thinking(text):
    return re.sub(r'<think>.*?</think>\s*', '', str(text or ''), flags=re.DOTALL | re.IGNORECASE).strip()


def clean_wikipedia_text(text):
    text = html.unescape(str(text or ''))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\[[^\]]*\]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_wikipedia_query_prompt(question):
    return f'''Decide whether Wikipedia would help answer this multiple-choice question.
Use Wikipedia only for external factual knowledge, named concepts, historical facts, definitions, people, places, or specialized terminology.
Return an empty query for self-contained arithmetic, algebra, probability, statistics, or logic questions.
Return exactly one valid JSON object and nothing else:
{{"query":"short Wikipedia search query or empty string","reason":"short reason"}}

Question:
{question_to_text(question)}

JSON:'''


def wikipedia_get_json(params):
    url = 'https://en.wikipedia.org/w/api.php?' + urllib.parse.urlencode(params)
    request = urllib.request.Request(url, headers={'User-Agent': 'PoliMillionaireSolver/1.0'})
    with urllib.request.urlopen(request, timeout=WIKIPEDIA_TIMEOUT_SECONDS) as response:
        return json.loads(response.read().decode('utf-8'))


def fetch_wikipedia_snippets(query):
    query = clean_wikipedia_text(query)[:120]
    if not query:
        return []
    search_data = wikipedia_get_json({
        'action': 'query',
        'list': 'search',
        'srsearch': query,
        'srlimit': max(1, min(WIKIPEDIA_MAX_SNIPPETS, 3)),
        'format': 'json',
    })
    results = search_data.get('query', {}).get('search', [])[:WIKIPEDIA_MAX_SNIPPETS]
    if not results:
        return []
    pageids = '|'.join(str(item.get('pageid')) for item in results if item.get('pageid'))
    extracts = {}
    if pageids:
        extract_data = wikipedia_get_json({
            'action': 'query',
            'prop': 'extracts',
            'exintro': 1,
            'explaintext': 1,
            'pageids': pageids,
            'format': 'json',
        })
        extracts = extract_data.get('query', {}).get('pages', {})

    snippets = []
    per_snippet = max(200, WIKIPEDIA_MAX_CONTEXT_CHARS // max(1, len(results)))
    for item in results:
        pageid = str(item.get('pageid', ''))
        title = clean_wikipedia_text(item.get('title', 'Wikipedia result'))
        extract = clean_wikipedia_text(extracts.get(pageid, {}).get('extract', ''))
        snippet = extract or clean_wikipedia_text(item.get('snippet', ''))
        if not snippet:
            continue
        title_url = urllib.parse.quote(title.replace(' ', '_'))
        snippets.append({'title': title, 'snippet': snippet[:per_snippet], 'url': f'https://en.wikipedia.org/wiki/{title_url}'})
    return snippets


def format_wikipedia_context(wiki_result):
    snippets = wiki_result.get('snippets', []) if isinstance(wiki_result, dict) else []
    if not snippets:
        return 'No Wikipedia context supplied.'
    lines = []
    used_chars = 0
    for item in snippets:
        title = clean_wikipedia_text(item.get('title', 'Wikipedia'))
        snippet = clean_wikipedia_text(item.get('snippet', ''))
        line = f'- {title}: {snippet}'
        if used_chars + len(line) > WIKIPEDIA_MAX_CONTEXT_CHARS:
            break
        lines.append(line)
        used_chars += len(line)
    return '\n'.join(lines) if lines else 'No Wikipedia context supplied.'


def run_wikipedia_tool(question):
    if not USE_WIKIPEDIA_TOOL:
        return {'ok': False, 'used': False, 'reason': 'Wikipedia tool disabled.'}
    raw_output = generate_text(build_wikipedia_query_prompt(question), max_new_tokens=WIKIPEDIA_QUERY_TOKENS, temperature=0.0)
    cleaned_output = strip_private_thinking(raw_output)
    request = extract_last_json_object(cleaned_output) or extract_last_json_object(raw_output)
    if not isinstance(request, dict):
        return {'ok': False, 'used': False, 'reason': 'Wikipedia query prompt did not return valid JSON.', 'raw_output': cleaned_output or raw_output}
    query = clean_wikipedia_text(request.get('query', ''))[:120]
    if not query:
        return {'ok': False, 'used': False, 'reason': str(request.get('reason', 'Wikipedia not needed.')), 'raw_output': cleaned_output or raw_output, 'request': request}
    try:
        snippets = fetch_wikipedia_snippets(query)
    except Exception as exc:
        return {'ok': False, 'used': True, 'reason': f'Wikipedia search failed: {type(exc).__name__}: {exc}', 'raw_output': cleaned_output or raw_output, 'request': request, 'query': query}
    return {'ok': bool(snippets), 'used': True, 'reason': str(request.get('reason', 'Wikipedia context retrieved.')), 'raw_output': cleaned_output or raw_output, 'request': request, 'query': query, 'snippets': snippets, 'context': format_wikipedia_context({'snippets': snippets})}


def build_reasoning_answer_prompt(question, wiki_result=None):
    return f'''Solve this conceptual multiple-choice question.
Use chain-of-thought reasoning privately, then return only the final structured result.
Do not reveal hidden scratchpad. Return concise public reasoning steps instead.
If Wikipedia context is supplied, use it only when it is directly relevant. Prefer facts stated in the question over retrieved context.
Return exactly one valid JSON object and nothing else. Do not use Markdown.

JSON schema:
{{"answer":"A","confidence":"low|medium|high","reason":"one concise final reason","reasoning_steps":["short step 1","short step 2"],"eliminated":{{"A":"short note","B":"short note","C":"short note","D":"short note"}}}}

Wikipedia context:
{format_wikipedia_context(wiki_result or {})}

Question:
{question_to_text(question)}

JSON:'''


def run_reasoning_answer(question, wiki_result=None):
    raw_output = generate_text(build_reasoning_answer_prompt(question, wiki_result), max_new_tokens=REASONING_TOKENS, temperature=MODEL_TEMPERATURE)
    cleaned_output = strip_private_thinking(raw_output)
    result = extract_last_json_object(cleaned_output) or extract_last_json_object(raw_output)
    letter = extract_letter(result.get('answer', '')) if isinstance(result, dict) else None
    if letter not in LETTERS:
        return {'ok': False, 'answer': None, 'confidence': 'low', 'reason': 'Reasoning solver did not return valid JSON.', 'raw_output': cleaned_output or raw_output}
    steps = result.get('reasoning_steps', [])
    if isinstance(steps, str):
        steps = [steps]
    if not isinstance(steps, list):
        steps = []
    steps = [str(step).strip() for step in steps if str(step).strip()][:4]
    return {'ok': True, 'answer': letter, 'confidence': str(result.get('confidence', 'medium')).strip().lower(), 'reason': str(result.get('reason', 'reasoning answer')).strip(), 'reasoning_steps': steps, 'eliminated': result.get('eliminated', {}), 'raw_output': cleaned_output or raw_output, 'source': 'llm:reasoning_solver'}


def build_structured_answer_prompt(question):
    return f'''Answer this multiple-choice question. Use step-by-step reasoning internally and eliminate wrong choices internally.
Return exactly one valid JSON object and nothing else. Do not use Markdown. Do not reveal chain-of-thought.

JSON schema:
{{"answer":"A","confidence":"low|medium|high","reason":"one concise sentence","eliminated":{{"A":"short note","B":"short note","C":"short note","D":"short note"}}}}

Question:
{question_to_text(question)}

JSON:'''


def run_structured_answer(question):
    raw_output = generate_text(build_structured_answer_prompt(question), max_new_tokens=STRUCTURED_TOKENS, temperature=MODEL_TEMPERATURE)
    result = extract_last_json_object(raw_output)
    letter = extract_letter(result.get('answer', '')) if isinstance(result, dict) else None
    if letter not in LETTERS:
        return {'ok': False, 'answer': None, 'confidence': 'low', 'reason': 'Structured LLM did not return valid JSON.', 'raw_output': raw_output}
    return {'ok': True, 'answer': letter, 'confidence': str(result.get('confidence', 'medium')).strip().lower(), 'reason': str(result.get('reason', 'structured answer')).strip(), 'eliminated': result.get('eliminated', {}), 'raw_output': raw_output, 'source': 'llm:structured_answer'}


def build_letter_only_prompt(question):
    return f'''Return only one character: A, B, C, or D. Do not explain.

{question_to_text(question)}

Answer:'''


def extract_strict_letter_output(raw_output):
    text = str(raw_output).strip().upper()
    if '<THINK>' in text or '</THINK>' in text:
        return None
    match = re.fullmatch(r'([ABCD])\s*[).:]*', text)
    return match.group(1) if match else None


def run_letter_only_fallback(question):
    raw_output = generate_text(build_letter_only_prompt(question), max_new_tokens=LETTER_TOKENS, temperature=0.0)
    letter = extract_strict_letter_output(raw_output)
    if letter not in LETTERS:
        return None
    return {'ok': True, 'answer': letter, 'confidence': 'medium', 'reason': 'Selected by strict letter-only fallback.', 'raw_output': raw_output, 'source': 'llm:letter_only'}


def build_letter_score_prompt(question):
    return f'''Answer the question with only A, B, C, or D.

{question_to_text(question)}

Answer:'''


def score_option_letters(question):
    prompt = make_prompt(build_letter_score_prompt(question))
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_PROMPT_TOKENS).to(model.device)
    with torch.inference_mode():
        logits = model(**inputs).logits[0, -1]
    scores = {}
    for letter in LETTERS:
        candidate_scores = []
        for candidate in [letter, ' ' + letter, '\n' + letter]:
            token_ids = tokenizer.encode(candidate, add_special_tokens=False)
            if token_ids:
                candidate_scores.append(float(logits[token_ids[-1]].detach().cpu()))
        scores[letter] = max(candidate_scores) if candidate_scores else float('-inf')
    return scores


def run_letter_score_fallback(question):
    scores = score_option_letters(question)
    letter = max(scores, key=scores.get)
    return {'ok': True, 'answer': letter, 'confidence': 'low', 'reason': 'Selected by direct A/B/C/D token scoring.', 'scores': {key: round(value, 4) for key, value in scores.items()}, 'source': 'llm:letter_score'}


def choose_answer(question):
    started = time.monotonic()
    if len(question.options) < 4:
        return question.options[0].id, 'A', {'ok': False, 'reason': 'fewer than four options'}

    python_result = {'ok': False, 'answer': None, 'reason': 'python solver skipped'}
    wiki_result = {'ok': False, 'used': False, 'reason': 'Wikipedia tool skipped'}
    reasoning_result = {'ok': False, 'answer': None, 'reason': 'reasoning solver skipped'}
    structured_result = {'ok': False, 'answer': None, 'reason': 'structured LLM skipped'}

    if USE_PYTHON_SOLVER:
        python_result = run_python_solver(question)
        if python_result.get('ok'):
            solver_step = {'step': 0, 'action': {'action': 'python_solver', 'answer': python_result.get('answer'), 'code': python_result.get('request', {}).get('code', '')}, 'observation': python_result.get('execution', {}), 'raw_output': python_result.get('raw_output', '')}
            trace_result = {'ok': True, 'reason': python_result.get('reason'), 'trace': [solver_step]}
            raw = {'elapsed_seconds': round(time.monotonic() - started, 2), 'used_python': True, 'used_wikipedia': False, 'used_reasoning': False, 'used_fallback': False, 'python_result': python_result, 'wiki_result': wiki_result, 'reasoning_result': reasoning_result, 'structured_result': structured_result, 'final_result': python_result, 'solver_result': trace_result}
            return option_id_for_letter_safe(question, python_result['answer']), python_result['answer'], raw

    wiki_result = run_wikipedia_tool(question)

    if USE_REASONING_SOLVER:
        reasoning_result = run_reasoning_answer(question, wiki_result)
        if reasoning_result.get('ok'):
            wiki_step = {'step': 0, 'action': {'action': 'wikipedia_tool', 'used': wiki_result.get('used'), 'query': wiki_result.get('query'), 'reason': wiki_result.get('reason')}, 'observation': {'context': wiki_result.get('context', ''), 'snippets': wiki_result.get('snippets', [])}, 'raw_output': wiki_result.get('raw_output', '')}
            solver_step = {'step': 1, 'action': {'action': 'reasoning_solver', 'answer': reasoning_result.get('answer'), 'confidence': reasoning_result.get('confidence'), 'reasoning_steps': reasoning_result.get('reasoning_steps', []), 'eliminated': reasoning_result.get('eliminated', {})}, 'raw_output': reasoning_result.get('raw_output', ''), 'observation': {'python_result': python_result, 'wiki_result': wiki_result}}
            trace_result = {'ok': True, 'reason': reasoning_result.get('reason'), 'trace': [wiki_step, solver_step]}
            raw = {'elapsed_seconds': round(time.monotonic() - started, 2), 'used_python': False, 'used_wikipedia': bool(wiki_result.get('used')), 'used_reasoning': True, 'used_fallback': False, 'python_result': python_result, 'wiki_result': wiki_result, 'reasoning_result': reasoning_result, 'structured_result': structured_result, 'final_result': reasoning_result, 'solver_result': trace_result}
            return option_id_for_letter_safe(question, reasoning_result['answer']), reasoning_result['answer'], raw

    if USE_STRUCTURED_LLM:
        structured_result = run_structured_answer(question)
        if structured_result.get('ok'):
            solver_step = {'step': 0, 'action': {'action': 'structured_answer', 'answer': structured_result.get('answer'), 'confidence': structured_result.get('confidence'), 'eliminated': structured_result.get('eliminated', {})}, 'raw_output': structured_result.get('raw_output', ''), 'observation': {'python_result': python_result, 'wiki_result': wiki_result, 'reasoning_result': reasoning_result}}
            trace_result = {'ok': True, 'reason': structured_result.get('reason'), 'trace': [solver_step]}
            raw = {'elapsed_seconds': round(time.monotonic() - started, 2), 'used_python': False, 'used_wikipedia': bool(wiki_result.get('used')), 'used_reasoning': False, 'used_fallback': False, 'python_result': python_result, 'wiki_result': wiki_result, 'reasoning_result': reasoning_result, 'structured_result': structured_result, 'final_result': structured_result, 'solver_result': trace_result}
            return option_id_for_letter_safe(question, structured_result['answer']), structured_result['answer'], raw

    fallback_result = run_letter_only_fallback(question)
    used_fallback = True
    if not fallback_result:
        fallback_result = run_letter_score_fallback(question)

    letter = fallback_result.get('answer') if fallback_result else 'A'
    if letter not in LETTERS:
        letter = 'A'
        fallback_result = {'ok': False, 'answer': letter, 'confidence': 'low', 'reason': 'No valid answer after all fallbacks.'}

    solver_step = {'step': 0, 'action': {'action': fallback_result.get('source', 'fallback'), 'answer': letter, 'confidence': fallback_result.get('confidence')}, 'raw_output': fallback_result.get('raw_output', ''), 'observation': {'python_result': python_result, 'wiki_result': wiki_result, 'reasoning_result': reasoning_result, 'structured_result': structured_result}}
    trace_result = {'ok': bool(fallback_result.get('ok')), 'reason': fallback_result.get('reason'), 'trace': [solver_step]}
    raw = {'elapsed_seconds': round(time.monotonic() - started, 2), 'used_python': False, 'used_wikipedia': bool(wiki_result.get('used')), 'used_reasoning': False, 'used_fallback': used_fallback, 'python_result': python_result, 'wiki_result': wiki_result, 'reasoning_result': reasoning_result, 'structured_result': structured_result, 'final_result': fallback_result, 'solver_result': trace_result}
    return option_id_for_letter_safe(question, letter), letter, raw


In [ ]:
TRACE_WIDTH = int(os.getenv('MILLIONAIRE_TRACE_WIDTH', '100'))
TRACE_JSON_CHARS = int(os.getenv('MILLIONAIRE_TRACE_JSON_CHARS', '900'))


def compact_json(value, max_chars=TRACE_JSON_CHARS):
    try:
        text = json.dumps(value, ensure_ascii=True, indent=2)
    except TypeError:
        text = str(value)
    if len(text) > max_chars:
        return text[:max_chars] + ' ...[truncated]'
    return text


def wrapped(text, width=TRACE_WIDTH, indent=''):
    text = str(text).strip()
    if not text:
        return ''
    lines = []
    for line in text.splitlines():
        if len(line) <= width:
            lines.append(indent + line)
        else:
            lines.append(textwrap.fill(line, width=width, initial_indent=indent, subsequent_indent=indent))
    return '\n'.join(lines)


def print_block(title, value=None):
    print(title, flush=True)
    if value is not None:
        print(wrapped(value, indent='  '), flush=True)


def print_agent_trace(raw, show_raw_model_output=False):
    print('', flush=True)
    print('================ Agent trace ================', flush=True)
    print('Elapsed seconds:', raw.get('elapsed_seconds'), flush=True)
    print('Used Python:', raw.get('used_python'), flush=True)
    print('Used Wikipedia:', raw.get('used_wikipedia'), flush=True)
    print('Used reasoning:', raw.get('used_reasoning'), flush=True)
    print('Used fallback:', raw.get('used_fallback'), flush=True)

    solver_result = raw.get('solver_result', {})
    print('Solver status:', solver_result.get('ok'), '| reason:', solver_result.get('reason'), flush=True)

    trace = solver_result.get('trace', [])
    if not trace:
        print('No solver trace was recorded.', flush=True)

    for step in trace:
        print('---------------------------------------------', flush=True)
        print('Step:', step.get('step'), flush=True)
        if step.get('error'):
            print('Error:', step.get('error'), flush=True)

        action = step.get('action')
        if action:
            display_action = {key: value for key, value in action.items() if not key.startswith('_')}
            print_block('Model action:', compact_json(display_action))

        if step.get('observation') is not None:
            print_block('Tool observation:', compact_json(step.get('observation')))

        if show_raw_model_output and step.get('raw_output'):
            print_block('Raw model output:', step.get('raw_output'))

    final = raw.get('final_result', {})
    print('---------------------------------------------', flush=True)
    print_block('Final result object:', compact_json(final))
    print('============== End agent trace ==============', flush=True)
    print('', flush=True)


def play_game(
    competition_id=COMPETITION_ID,
    max_questions=None,
    delay=0.2,
    show_trace=True,
    show_raw_model_output=False,
):
    game = client.game.start(competition_id=competition_id, mode='text')
    answered = 0

    while game.in_progress:
        if max_questions is not None and answered >= max_questions:
            break

        question = game.current_question
        if question is None:
            break

        print('', flush=True)
        print('=============================================', flush=True)
        print('Level:', game.current_level, flush=True)
        print(wrapped(question.text), flush=True)
        for index, option in enumerate(question.options):
            print(wrapped(f'{chr(65 + index)}) {option.text}'), flush=True)

        option_id, letter, raw = choose_answer(question)
        final = raw.get('final_result', {})
        print('Predicted:', letter, '| confidence:', final.get('confidence'), '| reason:', final.get('reason'), flush=True)

        if show_trace:
            print_agent_trace(raw, show_raw_model_output=show_raw_model_output)

        try:
            result = game.answer(option_id)
        except TimeoutError:
            print('Timed out', flush=True)
            break
        except RateLimitError:
            print('Rate limited, waiting...', flush=True)
            time.sleep(5)
            result = game.answer(option_id)

        answered += 1
        print('Correct:', result.correct, '| Earned:', result.earned_amount, flush=True)

        if result.game_over:
            break

        time.sleep(delay)

    print('Final earned:', game.earned_amount, flush=True)
    return game.earned_amount

In [ ]:
play_game(COMPETITION_ID, show_trace=True, show_raw_model_output=True)